# Performance Metrics

创造性思维指标分析，如流畅性、原创性等。

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# 输入文件路径
pickle_dir = Path('../../data/pickle')
user_msgs_file = pickle_dir / 'user_msgs.pkl'

scoring_dir = Path('../../data/analysis/scoring')
originality_file = scoring_dir / 'originality.csv'

# 输出文件路径
output_dir = Path('../../data/analysis/performance')
performance_file = output_dir / 'performance.csv'
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# 假设外部有类别编码和原创性分数
# 例如：categories_df 包含每个回答的类别标签，originality_df 包含每个回答的原创性分数
originality_df = pd.read_csv(originality_file, encoding='utf-8')
originality_df = originality_df[originality_df['trial_index'] != 0]  # 过滤掉练习试次

# 获得总体原创性的中位数（用于后续计算）
overall_median_originality = originality_df['originality'].median()
print(f"总体中位数原创性分数: {overall_median_originality:.2f}")

# 按被试和物品分组计算绩效
performance_list = []

grouped = originality_df.groupby(['participant_id', 'item_name', 'trial_index'])

for (pid, item, trial), group in grouped:
    # 流畅性：想法总数
    fluency = len(group)
    
    # 平均原创性
    originality = group['originality'].mean() if 'originality' in group else np.nan
    
    # 另一种原创性指标：超过总体中位数原创性的想法数量
    # 使用中位数而非平均数来衡量原创性，以减少极端值的影响
    above_median = (group['originality'] > overall_median_originality).sum()

    # 比率
    above_median_ratio = above_median / fluency if fluency > 0 else np.nan

    performance_list.append({
        'participant_id': pid,
        'item_name': item,
        'trial_index': trial,
        'fluency': fluency,
        'originality': originality,
        'above_median': above_median,
        'above_median_ratio': above_median_ratio
    })

performance_df = pd.DataFrame(performance_list)

In [ ]:
performance_df.describe()

In [ ]:
performance_df.to_csv(performance_file, index=False)